In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial
import scipy.spatial.distance

In [2]:
# Torch Load X_left.trc and X_right.trc, Y_left.trc and Y_right.trc
X_left = torch.load('X_left.trc').numpy()
X_right = torch.load('X_right.trc').numpy()
Y_left = torch.load('Y_left.trc').numpy()
Y_right = torch.load('Y_right.trc').numpy()
X_dmm = torch.load('X_dmm.trc').numpy()
Y_dmm = torch.load('Y_dmm.trc').numpy()

/tmp/ipykernel_1944956/3389405394.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_left = torch.load('X_left.trc').numpy()
/tmp/ipykernel_1944956/3389405394.py:3: Futur

In [3]:
def filter_left_shoulder_picks(X, Y):
    X_left, Y_left = X, Y
    start_config = X_left[:, -6:]
    end_config = Y_left[:, -6:]

    condition = (
        ((start_config[:, 0] > np.deg2rad(-50)) & (start_config[:, 1] > np.deg2rad(-90)) & (start_config[:, 2] < 0)) |
        ((start_config[:, 0] > np.deg2rad(-50)) & (start_config[:, 1] > np.deg2rad(-90)) & (start_config[:, 2] > 0)) |
        ((start_config[:, 0] < np.deg2rad(-50)) & (start_config[:, 1] > np.deg2rad(-90)) & (start_config[:, 2] > 0)) |
        ((start_config[:, 0] > np.deg2rad(-50)) & (start_config[:, 1] < np.deg2rad(-90)) & (start_config[:, 2] > 0))
    ) & (end_config[:, 0] > np.deg2rad(20))

    filtered_X, filtered_Y = X_left[condition], Y_left[condition]

    left_count = np.sum(condition)
    left_percentage = (left_count / len(X_left)) * 100
    
    # Print the info
    print(f"Number of left shoulder picks: {left_count}")
    print(f"Percentage of left shoulder picks: {left_percentage:.2f}%")

    return filtered_X, filtered_Y


import numpy as np

def filter_right_shoulder_picks(X, Y):
    start_config = X[:, -6:]
    end_config = Y[:, -6:]

    condition = (
        ((start_config[:, 0] < np.deg2rad(-50)) & (start_config[:, 1] > np.deg2rad(-90)) & (start_config[:, 2] < 0)) |
        ((start_config[:, 0] < np.deg2rad(-50)) & (start_config[:, 1] < np.deg2rad(-90)) & (start_config[:, 2] < 0)) |
        ((start_config[:, 0] > np.deg2rad(-50)) & (start_config[:, 1] < np.deg2rad(-90)) & (start_config[:, 2] < 0)) |
        ((start_config[:, 0] < np.deg2rad(-50)) & (start_config[:, 1] < np.deg2rad(-90)) & (start_config[:, 2] > 0))
    ) & (end_config[:, 0] < np.deg2rad(20))

    filtered_X, filtered_Y = X[condition], Y[condition]

    right_count = np.sum(condition)
    right_percentage = (right_count / len(X)) * 100
    
    # Print the info
    print(f"Number of right shoulder picks: {right_count}")
    print(f"Percentage of right shoulder picks: {right_percentage:.2f}%")

    return filtered_X, filtered_Y



In [6]:
from utils import filter_data_dist

# first filter the left shoulder and right shoulder picks
# then filter the data based on the distance
# for each X, Y data pair

def filter_pipeline (X, Y, r, name):
    filtered_X_left, filtered_Y_left = filter_left_shoulder_picks(X, Y)
    filtered_X_right, filtered_Y_right = filter_right_shoulder_picks(X, Y)
    filtered_X = np.concatenate([filtered_X_left, filtered_X_right], axis=0)
    filtered_Y = np.concatenate([filtered_Y_left, filtered_Y_right], axis=0)
    filtered_X, filtered_Y = filter_data_dist(filtered_X, filtered_Y, r)
    
    # Save the file and name after the input variable
    np.save(f'filtered_X_{name}.npy', filtered_X)
    np.save(f'filtered_Y_{name}.npy', filtered_Y)


In [7]:
filter_pipeline(X_left, Y_left, 1.75, 'left')

Number of left shoulder picks: 225457
Percentage of left shoulder picks: 69.13%
Number of right shoulder picks: 852
Percentage of right shoulder picks: 0.26%
30 0.9783702813409983 0.9783702813409983
29 0.9689937209744198 0.9904161435139602
28 0.9610134815672376 0.9917644054502672
27 0.9525692747526612 0.9912132275180932
26 0.9431396895395233 0.9901008929606865
25 0.9328086819348766 0.9890461532695218
24 0.9212492653849383 0.9876079449368318
23 0.9090182007785815 0.986723392825452
22 0.89581059524809 0.9854704718572421
21 0.8809194508393392 0.9833769052434272
20 0.8648926909667756 0.9818067817014446
19 0.8475624036162945 0.9799624999361375


In [8]:
filter_pipeline(X_right, Y_right, 1.75, 'right')

Number of left shoulder picks: 8263
Percentage of left shoulder picks: 2.35%
Number of right shoulder picks: 106049
Percentage of right shoulder picks: 30.21%
30 0.9763191965847855 0.9763191965847855
29 0.9665739379942613 0.9900183683526723
28 0.9572923227657638 0.9903974079336778
27 0.9462436139687871 0.9884583752170337
26 0.9335240394709217 0.9865578226261245
25 0.9188011757295821 0.9842287256472969
24 0.9021974945762474 0.981928972674474
23 0.883957939673875 0.9797831904743436
22 0.8637413394919169 0.9771294546102309
21 0.8421425572118413 0.9749939231891104
20 0.8179806144586745 0.9713089636116219
19 0.7912205192805655 0.9672851719159403


In [ ]:
filter_pipeline(X_dmm, Y_dmm, 1.5, 'dmm')

In [5]:
import json
import os
raw_task_sols_folder = "raw_task_sols_mixed"
for filename in os.listdir(raw_task_sols_folder):
    if filename.startswith("single_task_sols_") and filename.endswith(".json"):
        with open(os.path.join(raw_task_sols_folder, filename), 'r') as f:
            task_sols = json.load(f)
        # sub sample the task solutions and save as a new json file called sub_sampled_task_sols
        sub_sampled_task_sols = {}
        for task_id, task_sol in task_sols.items():
            sub_sampled_task_sols[task_id] = task_sol[::100] # sub sample every 10th element
        with open(f"sub_sampled_{filename}", 'w') as f:
            json.dump(sub_sampled_task_sols, f)